# Decision tree

Seed 42, hyperparameters chosen by a TPE search and features chosen by
out-of-fold permutation importance — neither by hand. Fitted and scored below,
then pointed at 2026, a season nobody has played yet.

Every model studied gets the same three cells: this heading, the fit, and the
2026 prediction. 2026 is what will say which of them was right.


In [3]:
import warnings

import numpy as np
import optuna
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from nfl_trees import metrics as metrics_mod
from nfl_trees.config import FeatureConfig
from nfl_trees.data import load_scores
from nfl_trees.features import build_dataset, make_preprocessor

SEED = 42
N_TRIALS = 80
HOLDOUT = 2025
METRICS = ["roc_auc", "accuracy", "log_loss", "brier"]

# The ten candidate features of notebook `02`, by the builder that makes them.
CANDIDATES = {
    "calendar": ["month", "week", "day", "playoff"],
    "win_rates": ["pct_home_win", "pct_away_win"],
    "drive_rates": [
        "home_pct_score_drive",
        "home_pct_allowed_drive",
        "away_pct_score_drive",
        "away_pct_allowed_drive",
    ],
}
CATEGORICAL = {"day"}
ALL_COLUMNS = [c for cols in CANDIDATES.values() for c in cols]


def config_for(columns):
    """A `FeatureConfig` over `columns`, dropping builders nothing survives from."""
    columns = [c for c in ALL_COLUMNS if c in set(columns)]
    return FeatureConfig(
        numeric=[c for c in columns if c not in CATEGORICAL],
        categorical=[c for c in columns if c in CATEGORICAL],
        builders=[b for b, cols in CANDIDATES.items() if set(cols) & set(columns)],
    )


# `drive_rates` folds every plays file, so the first run takes a few seconds.
FULL = config_for(ALL_COLUMNS)
games = load_scores()
X, y, meta = build_dataset(games, FULL, "home_win")
y = y.astype(int)
season = meta["Season"].astype(int)

# 2010 is the warm-up season the history rates have no previous season for, and
# the holdout is walled off from everything below: neither the hyperparameter
# search nor the feature selection is allowed to see 2025.
TUNING_SEASONS = [s for s in sorted(season.unique()) if 2013 <= s < HOLDOUT]


def fit_tree(train, params, features):
    """The pipeline a config would produce: repo preprocessor, then the tree."""
    pipe = Pipeline(
        [
            ("prep", make_preprocessor(features)),
            ("model", DecisionTreeClassifier(random_state=SEED, **params)),
        ]
    )
    return pipe.fit(X.loc[train, features.columns], y[train])


def cv_auc(params, features):
    """Rolling-origin CV: each season scored by a tree trained only on earlier ones.

    A single split would hand the search 280 games of noise to chase. Twelve
    seasons scored in sequence is the same discipline the season split enforces
    for a real run, applied twelve times.
    """
    folds = []
    for s in TUNING_SEASONS:
        model = fit_tree((season >= 2011) & (season < s), params, features)
        test = season == s
        folds.append(
            metrics_mod.compute(
                "classification",
                ["roc_auc"],
                y[test].to_numpy(),
                model.predict(X.loc[test, features.columns]),
                model.predict_proba(X.loc[test, features.columns])[:, 1],
            )["roc_auc"]
        )
    return float(np.mean(folds))


def tune(features, start_from=None):
    """TPE over five knobs, all bounded away from the pathological ends.

    `min_samples_leaf` starts at 10 because a leaf holding a handful of games
    returns probabilities of 0 or 1, and `log_loss` punishes every one of those
    that misses. `ccp_alpha` is cost-complexity pruning: the search can prune a
    deep tree back instead of only refusing to grow it. `start_from` seeds the
    study with a known-good point, so a second search cannot end up worse than
    the first one it is meant to improve on.
    """

    def objective(trial):
        return cv_auc(
            dict(
                criterion=trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
                max_depth=trial.suggest_int("max_depth", 2, 12),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 250, log=True),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 200, log=True),
                ccp_alpha=trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
            ),
            features,
        )

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        study = optuna.create_study(
            direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        if start_from is not None:
            study.enqueue_trial(start_from)
        study.optimize(objective, n_trials=N_TRIALS)
    return study.best_params, study.best_value


def permutation_table(params, features, n_repeats=10):
    """How much roc_auc each column is worth, measured where it has to be.

    Shuffling a column breaks its link to the outcome; whatever the model loses
    was what the column was carrying. Measured on the *held-out* season of every
    fold, never on the training rows: on the training set a tree can look like
    it depends on a column it only memorised.
    """
    per_season = []
    for s in TUNING_SEASONS:
        model = fit_tree((season >= 2011) & (season < s), params, features)
        test = season == s
        result = permutation_importance(
            model,
            X.loc[test, features.columns],
            y[test],
            scoring="roc_auc",
            n_repeats=n_repeats,
            random_state=SEED,
        )
        per_season.append(pd.Series(result.importances_mean, index=features.columns))

    folds = pd.DataFrame(per_season, index=TUNING_SEASONS)
    table = pd.DataFrame(
        {
            "mean": folds.mean(),
            "std_err": folds.std(ddof=1) / np.sqrt(len(folds)),
            "seasons_up": (folds > 0).sum(),
        }
    )
    table["t"] = table["mean"] / table["std_err"].replace(0.0, np.nan)
    return table.sort_values("mean", ascending=False)


# -- 1. tune on every candidate feature -------------------------------------- #
params_full, cv_full = tune(FULL)
print(f"TPE search, {N_TRIALS} trials over {len(TUNING_SEASONS)} folds (2013-2024)")
print("params  : " + "  ".join(f"{k}={v}" for k, v in params_full.items()))
print(f"cv auc  : {cv_full:.4f}   on all {len(FULL.columns)} features")

# -- 2. ask each feature what it is worth ------------------------------------ #
importance = permutation_table(params_full, FULL)
print("\npermutation importance, out of fold: roc_auc lost when the column is shuffled")
print(importance.round(5).to_string())


# A feature stays when its mean clears its own standard error *and* more than
# one season carried it. Two conditions rather than one because of `week`:
# positive in a single season out of twelve and exactly zero in the other
# eleven, which makes its mean and its standard error algebraically the same
# number -- `mean > 0` and `mean > std_err` would both come down to the last bit
# of a float rather than to anything about football.
#
# Cross-validation is deliberately not the judge here. Refitting without a
# column moves the tree even when that column was never split on, because ties
# between equally good splits break differently, and that wobble is worth more
# roc_auc than any of these features: a CV-guided elimination on these folds
# drops `pct_home_win` (t = 2.6) and keeps `day` (importance exactly zero).
# Permutation importance is read off one fitted model per fold, so it never
# refits and never sees that noise. It decides; CV only checks the bill below.
survives = (importance["mean"] > importance["std_err"]) & (importance["seasons_up"] > 1)
KEEP = list(importance.index[survives])
DROPPED = list(importance.index[~survives])
print(f"\nkept    : {', '.join(KEEP)}")
print(f"dropped : {', '.join(DROPPED)}")

# -- 3. retune on the survivors ---------------------------------------------- #
FEATURES = config_for(KEEP)
PARAMS, cv_reduced = tune(FEATURES, start_from=params_full)
cv_cut = cv_auc(params_full, FEATURES)
print(f"\nretuned on the {len(KEEP)} survivors, builders now {FEATURES.builders}")
print("params  : " + "  ".join(f"{k}={v}" for k, v in PARAMS.items()))
print(f"cv auc  : {cv_reduced:.4f}   ({cv_reduced - cv_full:+.4f} against all {len(FULL.columns)})")
print(f"the cut : {cv_cut:.4f} with the step 1 params ({cv_cut - cv_full:+.4f}) -- what the four")
print("          dropped columns were worth to a tree that never split on them")

# -- 4. the holdout neither step was allowed to see -------------------------- #
train, test = season.between(2011, 2024), season.eq(HOLDOUT)
report = pd.DataFrame(
    {
        label: {
            "cv_roc_auc": round(cv, 4),
            **{
                k: round(v, 3)
                for k, v in metrics_mod.compute(
                    "classification",
                    METRICS,
                    y[test].to_numpy(),
                    fit_tree(train, params, features).predict(X.loc[test, features.columns]),
                    fit_tree(train, params, features).predict_proba(
                        X.loc[test, features.columns]
                    )[:, 1],
                ).items()
            },
        }
        for label, params, features, cv in (
            (f"all {len(FULL.columns)}", params_full, FULL, cv_full),
            (f"kept {len(KEEP)}", PARAMS, FEATURES, cv_reduced),
        )
    }
).T
print(f"\ntrain 2011-2024 ({int(train.sum())} games), holdout {HOLDOUT} ({int(test.sum())} games)")
print(report.to_string())
print(f"always-home : accuracy {y[test].mean():.3f}   <- the bar to clear")

# What the next cell predicts with: the kept features, refit on every game played.
model = fit_tree(season.between(2011, HOLDOUT), PARAMS, FEATURES)
print(f"\nrefit on 2011-{HOLDOUT} ({int(season.between(2011, HOLDOUT).sum())} games) to predict 2026")


TPE search, 80 trials over 12 folds (2013-2024)
params  : criterion=log_loss  max_depth=5  min_samples_leaf=98  min_samples_split=37  ccp_alpha=2.235791197815289e-06
cv auc  : 0.6148   on all 10 features

permutation importance, out of fold: roc_auc lost when the column is shuffled
                           mean  std_err  seasons_up        t
away_pct_score_drive    0.05277  0.00770          12  6.85530
home_pct_allowed_drive  0.02285  0.00928           9  2.46233
home_pct_score_drive    0.02034  0.00647          10  3.14523
pct_home_win            0.01026  0.00396           7  2.59198
away_pct_allowed_drive  0.00757  0.00344           9  2.20309
pct_away_win            0.00236  0.00174           3  1.35331
week                    0.00013  0.00013           1  1.00000
month                   0.00000  0.00000           0      NaN
playoff                 0.00000  0.00000           0      NaN
day                     0.00000  0.00000           0      NaN

kept    : away_pct_score_drive, ho

In [4]:
from nfl_trees.data import canonical_team
from nfl_trees.features import apply_builders

DIVISIONS = {
    "AFC East": ["BUF", "MIA", "NE", "NYJ"],
    "AFC North": ["BAL", "CIN", "CLE", "PIT"],
    "AFC South": ["HOU", "IND", "JAX", "TEN"],
    "AFC West": ["DEN", "KC", "LAC", "LV"],
    "NFC East": ["DAL", "NYG", "PHI", "WAS"],
    "NFC North": ["CHI", "DET", "GB", "MIN"],
    "NFC South": ["ATL", "CAR", "NO", "TB"],
    "NFC West": ["ARI", "LAR", "SEA", "SF"],
}
DIVISION = {team: div for div, teams in DIVISIONS.items() for team in teams}
TEAMS = sorted(DIVISION)


def home_win_proba(frame):
    """Probability the home team wins, one per row of a scores-shaped frame.

    `build_dataset` cannot be used here: it drops every row with a missing
    target, which is all of them when the games have not been played. The
    builders still work -- for any 2026 week the history rates read 2025 in
    full, which is exactly what a forecast made today has to go on.
    """
    design = apply_builders(frame, FEATURES.builders)[FEATURES.columns]
    numeric = design[FEATURES.numeric].apply(pd.to_numeric, errors="coerce").astype(float)
    categorical = design[FEATURES.categorical].astype(object).where(
        design[FEATURES.categorical].notna(), np.nan
    )
    return model.predict_proba(pd.concat([numeric, categorical], axis=1)[FEATURES.columns])[:, 1]


# -- regular season ---------------------------------------------------------- #
schedule = load_scores([2026], statuses=("TBD",), include_postseason=False)
schedule = schedule.assign(
    home=canonical_team(schedule["HomeTeam"]), away=canonical_team(schedule["AwayTeam"])
)
schedule["p_home"] = home_win_proba(schedule)
schedule["winner"] = np.where(schedule["p_home"] >= 0.5, schedule["home"], schedule["away"])

# Two readings of the same 272 probabilities. `W-L` is the record the picks add
# up to, and a team favoured every single week goes 17-0 in it; `exp` sums the
# probabilities instead, which is the win total the model would actually bet on.
wins = schedule["winner"].value_counts().reindex(TEAMS).fillna(0)
played = (
    schedule["home"].value_counts()
    .add(schedule["away"].value_counts(), fill_value=0)
    .reindex(TEAMS)
)
expected = (
    schedule.groupby("home")["p_home"].sum()
    .add(schedule.assign(p=1 - schedule["p_home"]).groupby("away")["p"].sum(), fill_value=0)
    .reindex(TEAMS)
)

standings = pd.DataFrame(
    {
        "division": [DIVISION[team] for team in TEAMS],
        "W": wins.astype(int),
        "L": (played - wins).astype(int),
        "exp": expected.round(1),
    },
    index=TEAMS,
).sort_values(["W", "exp"], ascending=False)
standings["record"] = standings["W"].astype(str) + "-" + standings["L"].astype(str)

print(f"2026 regular season, {len(schedule)} games predicted")
print("W-L is the record the picks add up to, exp is the summed probabilities\n")
for division in DIVISIONS:
    block = standings[standings["division"] == division]
    print(
        f"{division:<10}  "
        + "   ".join(f"{t:<3} {r.record:>5} ({r.exp:>4.1f})" for t, r in block.iterrows())
    )


# -- playoffs ---------------------------------------------------------------- #
ROUNDS = {
    "wild_card": ("WILD CARD WEEKEND", "January 10th"),
    "divisional": ("DIVISIONAL PLAYOFFS", "January 17th"),
    "championship": ("CONFERENCE CHAMPIONSHIPS", "January 24th"),
    "super_bowl": ("SUPER BOWL", "February 7th"),
}


def seeds_of(conference):
    """Seeds 1-7: the four division winners by record, then the three best left."""
    table = standings[standings["division"].str.startswith(conference)]
    champions = [block.index[0] for _, block in table.groupby("division", sort=False)]
    won_division = table.index.isin(champions)
    ranked = list(table.index[won_division]) + list(table.index[~won_division][:3])
    return list(enumerate(ranked, start=1))


def round_frame(pairs, round_key):
    week, date = ROUNDS[round_key]
    return pd.DataFrame(
        [
            {
                "Season": 2026, "Week": week, "GameStatus": "TBD", "GameSlot": "Sunday",
                "GameDate": date, "AwayTeam": away, "AwayScore": np.nan,
                "HomeTeam": home, "HomeScore": np.nan,
            }
            for (_, home), (_, away) in pairs
        ]
    )


def play_round(pairs, round_key, label, neutral=False):
    """Run one round. `pairs` is (host, visitor) as (seed, team); returns the winners.

    On a neutral field nobody hosts, and the model has no way to be told that --
    the four venue features are half of what it reads. So the Super Bowl is
    predicted twice, once with each team as the home side, and the two averaged.
    """
    p_first = home_win_proba(round_frame(pairs, round_key))
    if neutral:
        flipped = home_win_proba(round_frame([(b, a) for a, b in pairs], round_key))
        p_first = (p_first + (1 - flipped)) / 2

    print(f"\n{label}")
    winners = []
    for (first, second), p in zip(pairs, p_first):
        if abs(p - 0.5) < 1e-9:
            # Both teams landed in the same leaf: a tree with a few dozen leaves
            # cannot separate every pair. Decided on the regular season the model
            # just predicted rather than on the order the pair happens to be in.
            winner = max((first, second), key=lambda seed: standings.loc[seed[1], "exp"])
            note = "   (same leaf: settled on expected wins)"
        else:
            winner, note = (first if p > 0.5 else second), ""
        prob = p if winner == first else 1 - p
        print(
            f"  ({second[0]}) {second[1]:<3} {'vs' if neutral else 'at'} ({first[0]}) {first[1]:<3}"
            f"   ->  {winner[1]:<3} {prob:6.1%}{note}"
        )
        winners.append(winner)
    return winners


print("\n\n2026 playoffs, predicted -- the better seed hosts every round")
finalists = {}
for conference in ("AFC", "NFC"):
    seeds = seeds_of(conference)
    print("\n" + conference + " seeds: " + "  ".join(f"{n}.{team}" for n, team in seeds))

    # Seed 1 sits out the wild card round; every later round re-seeds, so the
    # best team left always hosts the worst one left.
    alive = sorted(
        [seeds[0]]
        + play_round(
            [(seeds[1], seeds[6]), (seeds[2], seeds[5]), (seeds[3], seeds[4])],
            "wild_card",
            f"{conference} wild card   ({seeds[0][1]} on a bye)",
        )
    )
    alive = sorted(
        play_round(
            [(alive[0], alive[3]), (alive[1], alive[2])],
            "divisional",
            f"{conference} divisional",
        )
    )
    finalists[conference] = play_round(
        [(alive[0], alive[1])], "championship", f"{conference} championship"
    )[0]

champion = play_round(
    [(finalists["AFC"], finalists["NFC"])],
    "super_bowl",
    "Super Bowl   (neutral field: predicted from both sides and averaged)",
    neutral=True,
)[0]
print(f"\n  champion: {champion[1]}")


2026 regular season, 272 games predicted
W-L is the record the picks add up to, exp is the summed probabilities

AFC East    NE   15-2 (11.2)   BUF   8-9 ( 8.8)   MIA  3-14 ( 7.9)   NYJ  1-16 ( 5.6)
AFC North   CIN  12-5 ( 9.5)   BAL  10-7 ( 9.3)   PIT  5-12 ( 7.9)   CLE  4-13 ( 5.6)
AFC South   HOU  16-1 (10.2)   JAX  15-2 (10.5)   IND  10-7 ( 9.6)   TEN  1-16 ( 4.6)
AFC West    DEN  14-3 (10.2)   KC   12-5 (10.1)   LAC  11-6 ( 8.8)   LV   1-16 ( 4.6)
NFC East    PHI  11-6 ( 8.2)   DAL   9-8 ( 9.6)   NYG   8-9 ( 8.9)   WAS  2-15 ( 7.7)
NFC North   DET  12-5 ( 9.7)   MIN  11-6 ( 8.4)   CHI  10-7 ( 9.0)   GB   10-7 ( 8.4)
NFC South   TB    9-8 ( 8.7)   ATL  7-10 ( 8.2)   NO   5-12 ( 7.5)   CAR  1-16 ( 7.1)
NFC West    SEA  15-2 ( 9.8)   LAR  14-3 (10.6)   SF    8-9 ( 9.1)   ARI  2-15 ( 6.5)


2026 playoffs, predicted -- the better seed hosts every round

AFC seeds: 1.HOU  2.NE  3.DEN  4.CIN  5.JAX  6.KC  7.LAC

AFC wild card   (HOU on a bye)
  (7) LAC at (2) NE    ->  NE   77.0%
  (6) K